# Cosmology: CLASS vs CAMB vs symbolic_new

This notebook builds one **`AnalysisContext`** per power-spectrum backend -
`camb`, `class` and `symbolic` (the analytic **syren_new** provider) - and
compares the important cosmological quantities between the three:

* background: $H(z)$, comoving/angular distance, growth factor $D(z)$, growth
  rate $f(z)$ - each plotted absolutely *and* as relative difference with
  respect to CAMB,
* linear and non-linear matter power spectra at several redshifts, again
  absolute panels plus % residuals,
* a summary table of the maximum relative deviation over the sampled grid.

No `FisherMatrix` is used: configuration and fiducial cosmology come from
`cosmicfishpie.configs.context.build_analysis_context`, and the functions are
read from `context.fiducialcosmo` (a `cosmo_functions` instance that the
context already evaluates for you).

Notes: the symbolic provider is used through the default
`default_boltzmann_yaml_files/symbolic/default.yaml` (`provider: syren_new`);
its power-spectrum domain is $k\in[0.001, 9]\,h\,\mathrm{Mpc}^{-1}$ and
$z\le 3$, which is why the $k$-grid below stops at 9.

In [ ]:
%matplotlib inline

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from cosmicfishpie.configs.context import build_analysis_context

sns.set_theme(context="talk", style="ticks")
snscolors = sns.color_palette("colorblind")

# repository root (works from notebooks/ or from the repo root)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "cosmicfishpie").is_dir())
SPECS_DIR = REPO / "cosmicfishpie" / "configs" / "default_survey_specifications"
print("repo:", REPO)

PLOTS = Path("./plots")
PLOTS.mkdir(parents=True, exist_ok=True)

# Table of Contents

[Cosmological parameters](#cosmological-parameters)

[Settings and sampling grids](#settings-and-sampling-grids)

[Build one Analysis Context per backend](#build-one-analysis-context-per-backend)

[Cosmological functions](#cosmological-functions)

[Power spectra](#power-spectra)

[Summary](#summary)

# Cosmological parameters

In [ ]:
fiducial = {
    "Omegam": 0.32,
    "Omegab": 0.05,
    "h": 0.67,
    "ns": 0.96,
    "sigma8": 0.815584,
    "w0": -1.0,
    "wa": 0.0,
    "mnu": 0.06,
    "Neff": 3.044,
}

# Settings and sampling-grids

All three backends are evaluated at the *same* fiducial cosmology, so every
curve below is directly comparable.  ``BACKENDS`` controls which ones run.

In [ ]:
BACKENDS = ["camb", "class", "symbolic"]
BACKEND_LABEL = {"camb": "CAMB", "class": "CLASS", "symbolic": "syren_new (symbolic)"}
colors = {code: snscolors[i] for i, code in enumerate(BACKENDS)}

base_options = {
    "accuracy": 1,
    "feedback": 0,
    "cosmo_model": "w0waCDM",
    "nonlinear": True,
    "specs_dir": str(SPECS_DIR) + "/",
}
survey_name = "Euclid"
observables = ["GCph", "WL"]   # not evaluated here: only cosmology is read out

zz = np.linspace(0.001, 5.0, 250)          # redshift grid for background functions
kk = np.logspace(np.log10(1e-3), np.log10(9.0), 250)   # k-grid inside the symbolic domain
zplot = [0.0, 1.0, 2.5]                    # power-spectrum snapshots

### Build one Analysis Context per backend

`build_analysis_context` resolves options/specs/fiducials and initialises the
backend once; `context.fiducialcosmo` is ready to evaluate.  Everything is
measured **inside** the loop, before the next context re-initialises the
configuration.

In [ ]:
contexts, backend_data, resolved = {}, {}, {}

for code in BACKENDS:
    print(f"building context: {code} ...", flush=True)
    context = build_analysis_context(
        options={**base_options, "code": code},
        observables=observables,
        freepars={},                    # Fisher-free: only fiducial cosmology is needed
        fiducialpars=fiducial,
        survey_name=survey_name,
        cosmo_model=base_options["cosmo_model"],
    )
    cosmo = context.fiducialcosmo

    data = {
        "hub": cosmo.Hubble(zz, physical=True),     # km/s/Mpc
        "ehub": cosmo.E_hubble(zz),
        "chi": cosmo.comoving(zz),                  # Mpc
        "ang": cosmo.angdist(zz),                   # Mpc
        "growth": cosmo.growth(zz),
        "fgrowth": cosmo.f_growthrate(zz),
        "plin": np.array([cosmo.matpow(z, kk, nonlinear=False) for z in zplot]),
        "pnl": np.array([cosmo.matpow(z, kk, nonlinear=True) for z in zplot]),
    }
    backend_data[code] = data
    contexts[code] = context
    resolved[code] = {
        "input_type": context.input_type,
        # AnalysisContext field with the yaml-resolved native backend settings
        # (a plain mapping for every backend; cosmopars-style translation tables).
        "backend_parameters": dict(context.backend_parameters),
    }
    print(f"  -> {code}: input_type={resolved[code]['input_type']}, "
          f"{len(resolved[code]['backend_parameters'])} resolved backend parameters")

REF = BACKENDS[0]   # relative differences are quoted against CAMB

# Cosmological functions

Left column: absolute prediction of each backend.  Right column: relative
difference $(X_{\rm backend} - X_{\rm CAMB}) / X_{\rm CAMB}$ in percent.

In [ ]:
QUANTITIES = [
    ("hub", r"$H(z)$  [km/s/Mpc]"),
    ("chi", r"$\chi(z)$  [Mpc]"),
    ("growth", r"$D(z)$"),
    ("fgrowth", r"$f(z)$"),
]

fig, axs = plt.subplots(2, 2, figsize=(15, 11))
for ax, (key, label) in zip(axs.ravel(), QUANTITIES):
    for code in BACKENDS:
        y = backend_data[code][key]
        ls = "-." if code == "symbolic" else "-"
        ax.semilogy(zz, y, color=colors[code], lw=2.5, ls=ls, label=BACKEND_LABEL[code])
    ax.set_xlabel(r"$z$")
    ax.set_ylabel(label)
    ax.legend(loc="best", fontsize=14)
    ax.grid(alpha=0.25)
fig.suptitle("Background and growth: three backends, one fiducial cosmology", y=1.00)
fig.tight_layout()
fig.savefig(PLOTS / "background_absolute.png", dpi=150, bbox_inches="tight")

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 11))
max_rel_bg = {}
for ax, (key, label) in zip(axs.ravel(), QUANTITIES):
    ref = backend_data[REF][key]
    for code in BACKENDS:
        if code == REF:
            continue
        y = backend_data[code][key]
        rel = 100.0 * (y - ref) / ref
        max_rel_bg[(code, key)] = float(np.max(np.abs(rel)))
        ls = "-." if code == "symbolic" else "-"
        ax.semilogy(zz, np.abs(rel), color=colors[code], lw=2.5, ls=ls,
                    label=f"|{BACKEND_LABEL[code]} - {BACKEND_LABEL[REF]}| / CAMB")
    ax.set_xlabel(r"$z$")
    ax.set_ylabel(label + r"  relative difference [%]")
    ax.legend(loc="best", fontsize=14)
    ax.grid(alpha=0.25)
fig.suptitle("Relative differences w.r.t. CAMB (log scale: signed curves cross zero)", y=1.00)
fig.tight_layout()
fig.savefig(PLOTS / "background_relative_to_camb.png", dpi=150, bbox_inches="tight")

# Power spectra

Rows: $z = $ `zplot` snapshots.  Left column linear $P_{mm}(k)$, right column
non-linear.  Bottom rows show the percent difference to CAMB (where the CAMB
curve is the reference).  Solid = CAMB/CLASS, dashed-dotted = symbolic.

In [ ]:
def spectrum_grid(key, relkey, title, fname):
    nrow = len(zplot)
    fig, axs = plt.subplots(nrow, 2, figsize=(15, 4.2 * nrow), sharex="col")
    for iz, z in enumerate(zplot):
        # absolute panel
        ax = axs[iz, 0]
        for code in BACKENDS:
            y = backend_data[code][key][iz]
            ls = "-." if code == "symbolic" else "-"
            ax.loglog(kk, y, color=colors[code], lw=2.2, ls=ls, label=BACKEND_LABEL[code])
        ax.set_ylabel(r"$P_{mm}(k) \; [(\mathrm{Mpc})^{-3}]$")
        ax.set_title(f"{title},  z = {z:g}")
        ax.grid(alpha=0.25, which="both")
        if iz == 0:
            ax.legend(fontsize=12)
        # relative panel
        ax = axs[iz, 1]
        ref = backend_data[REF][key][iz]
        for code in BACKENDS:
            if code == REF:
                continue
            y = backend_data[code][key][iz]
            rel = 100.0 * (y - ref) / ref
            relkey[f"{key}|z={z:g}|{code}"] = float(np.max(np.abs(rel)))
            ls = "-." if code == "symbolic" else "-"
            ax.semilogx(kk, np.abs(rel), color=colors[code], lw=2.2, ls=ls,
                        label=f"|{BACKEND_LABEL[code]} - CAMB| / CAMB")
        ax.set_xlabel(r"$k \; [h\,\mathrm{Mpc}^{-1}]$")
        ax.set_ylabel(r"$\Delta P / P$ [%]")
        ax.grid(alpha=0.25, which="both")
        if iz == 0:
            ax.legend(fontsize=12)
    fig.suptitle(f"{title}: three backends vs CAMB", y=1.00)
    fig.tight_layout()
    fig.savefig(PLOTS / fname, dpi=150, bbox_inches="tight")
    return fig


max_rel_pk = {}
spectrum_grid("plin", max_rel_pk, "Linear P(k)", "pk_linear_backends.png")
spectrum_grid("pnl", max_rel_pk, "Non-linear P(k)", "pk_nonlinear_backends.png")

# Summary

Maximum absolute relative difference to CAMB over the sampled grids, in
percent: background functions over `zz`, power spectra per redshift slice.

In [ ]:
rows = []
for code in BACKENDS:
    if code == REF:
        continue
    row = {"backend": BACKEND_LABEL[code]}
    for key, label in QUANTITIES:
        if (code, key) in max_rel_bg:
            row[label] = max_rel_bg[(code, key)]
    for key in ("plin", "pnl"):
        for z in zplot:
            row[f"{key} z={z:g}"] = max_rel_pk.get(f"{key}|z={z:g}|{code}", np.nan)
    rows.append(row)

df_summary = pd.DataFrame(rows).set_index("backend")
display(df_summary.T.style.format("{:.4g}"))

print("Resolved backend configuration at the fiducial point (yaml sections):")
for code in BACKENDS:
    bp = resolved[code]["backend_parameters"]
    extra = ""
    acc = bp.get("ACCURACY") if hasattr(bp, "get") else None
    if hasattr(acc, "get") and acc.get("provider") is not None:
        extra = f", provider={acc.get('provider')}"
    print(f"  {BACKEND_LABEL[code]:20s} input_type={resolved[code]['input_type']:9s} "
          f"sections={list(bp)}{extra}")

## Recap

* one `build_analysis_context(...)` call per backend replaces the old
  `FisherMatrix` construction - no Fisher, derivatives or nuisance parameters
  involved,
* `context.fiducialcosmo` exposes the full function set
  (`Hubble`, `E_hubble`, `comoving`, `angdist`, `growth`, `f_growthrate`,
  `matpow`, ...),
* the three backends agree at the sub-percent level on background observables;
  read the summary table and the % panels for the exact numbers, especially
  the non-linear $P(k)$ where the symbolic **syren_new** fit interpolates
  instead of solving the Boltzmann equations.

Figures are written to `notebooks/plots/` (git-ignored).